# ISL CSLR — Fixed Pipeline (Word → Sentence → Video curriculum)

Kept: `ISL_Conformer` (multi-stream, CTC) as the **single** model.
Canonical feature schema for **all 3 stages**: hands(126) + arms(12) + face(21) + flags(4) = **163-dim**, identical extraction function reused everywhere so weights transfer across stages.

Fixes applied vs. original notebook:
- One shared `extract_frame_features()` used for word images, sentence frames, and video (was 3 different schemas before).
- `ISL_Conformer` is the model actually instantiated/trained (previously dead code).
- Split real videos **before** synthetic oversampling, grouped by source file (prevents synth-sibling leakage across train/val).
- Scale normalization uses **per-frame** arm-based scale (not a sequence-mean), so dropped-tracking frames don't distort the whole clip.
- Sentence-level `Dataset` added (multi-word CTC targets) — this did not exist before.
- WER / sequence edit-distance metric added for sentence-level eval.


## 1. Download dataset

In [ ]:
import os
from pathlib import Path

!pip install --upgrade --quiet kaggle

if not (os.path.exists('/root/.kaggle/kaggle.json') or 'KAGGLE_USERNAME' in os.environ):
    print("⚠️ WARNING: Kaggle credentials not found! Upload kaggle.json to ~/.kaggle/ or set env vars.")
else:
    print("✅ Kaggle credentials detected.")

DATASET_SLUG = "drblack00/isl-csltr-indian-sign-language-dataset"
DATASET_PATH = "/content/isl_csltr_dataset"
os.makedirs(DATASET_PATH, exist_ok=True)

print(f"\nDownloading {DATASET_SLUG}...")
!kaggle datasets download -d {DATASET_SLUG} -p {DATASET_PATH} --unzip --force

dataset_dir = Path(DATASET_PATH)
files_only = [f for f in dataset_dir.rglob('*') if f.is_file()]
if files_only:
    total_size_mb = sum(f.stat().st_size for f in files_only) / (1024 * 1024)
    print(f"\n✅ Downloaded {len(files_only)} files, {total_size_mb:.2f} MB")
    for item in dataset_dir.iterdir():
        if item.is_dir():
            n = len([f for f in item.rglob('*') if f.is_file()])
            print(f"  ├── {item.name}/ ({n} files)")
else:
    print("\n❌ Download failed. Check Kaggle permissions or dataset availability.")


## 2. Install MediaPipe + download task models

In [ ]:
!pip install --quiet mediapipe

import os, urllib.request

models = {
    "hand_landmarker.task": "https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/latest/hand_landmarker.task",
    "pose_landmarker.task": "https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_lite/float16/latest/pose_landmarker_lite.task",
    "face_landmarker.task": "https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/latest/face_landmarker.task",
}
print("📥 Downloading MediaPipe task models...")
for name, url in models.items():
    if not os.path.exists(name):
        print(f"Fetching {name}...")
        urllib.request.urlretrieve(url, name)
print("✅ All models downloaded!")


## 3. Model — `ISL_Conformer` (the ONLY model used from here on)
163-dim input: `hands(126) | arms(12) | face(21) | flags(4)`. Multi-stream embedding → MBConv (local spatial) → Conformer stack → dense skip fusion → CTC classifier.

In [ ]:
import logging
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

logger = logging.getLogger("isl_conformer")

# ============================================================================
# 1. UTILS & MASKS
# ============================================================================
def lengths_to_padding_mask(lengths: torch.Tensor, max_len: int) -> torch.Tensor:
    """Returns a (B, T) bool mask where True == PADDING."""
    batch_size = lengths.size(0)
    arange = torch.arange(max_len, device=lengths.device).unsqueeze(0).expand(batch_size, -1)
    return arange >= lengths.unsqueeze(1)

class Transpose(nn.Module):
    def __init__(self, dim0, dim1):
        super().__init__()
        self.dim0, self.dim1 = dim0, dim1
    def forward(self, x):
        return x.transpose(self.dim0, self.dim1)

# ============================================================================
# 2. MULTI-STREAM EMBEDDING
# ============================================================================
class MultiStreamEmbedding(nn.Module):
    def __init__(self, d_model=256):
        super().__init__()
        # Hands (126 + 2 flags = 128)
        self.hand_proj = nn.Sequential(nn.LayerNorm(128), nn.Linear(128, d_model // 2))
        # Arms (12 + 1 flag = 13)
        self.arm_proj = nn.Sequential(nn.LayerNorm(13), nn.Linear(13, d_model // 4))
        # Face (21 + 1 flag = 22)
        self.face_proj = nn.Sequential(nn.LayerNorm(22), nn.Linear(22, d_model // 4))

    def forward(self, x):
        hands = x[:, :, 0:126]
        arms  = x[:, :, 126:138]
        face  = x[:, :, 138:159]
        flags = x[:, :, 159:163]

        hand_stream = torch.cat([hands, flags[:, :, 0:2]], dim=-1)
        arm_stream  = torch.cat([arms, flags[:, :, 2:3]], dim=-1)
        face_stream = torch.cat([face, flags[:, :, 3:4]], dim=-1)

        h_emb = self.hand_proj(hand_stream)
        a_emb = self.arm_proj(arm_stream)
        f_emb = self.face_proj(face_stream)

        return torch.cat([h_emb, a_emb, f_emb], dim=-1)  # (B, T, 256)

# ============================================================================
# 3. MBCONV (LOCAL SPATIAL)
# ============================================================================
class SqueezeExcitation1D(nn.Module):
    def __init__(self, channels, reduction=4):
        super().__init__()
        self.se = nn.Sequential(
            nn.AdaptiveAvgPool1d(1),
            nn.Conv1d(channels, channels // reduction, 1),
            nn.SiLU(),
            nn.Conv1d(channels // reduction, channels, 1),
            nn.Sigmoid(),
        )
    def forward(self, x):
        return x * self.se(x)

class EfficientNet1DBlock(nn.Module):
    def __init__(self, in_channels, out_channels, expansion=2):
        super().__init__()
        mid_channels = in_channels * expansion
        self.expand = nn.Conv1d(in_channels, mid_channels, kernel_size=1)
        self.bn_expand = nn.BatchNorm1d(mid_channels)

        self.depthwise = nn.Conv1d(mid_channels, mid_channels, kernel_size=3, padding=1, groups=mid_channels)
        self.bn_depthwise = nn.BatchNorm1d(mid_channels)

        self.se = SqueezeExcitation1D(mid_channels)
        self.project = nn.Conv1d(mid_channels, out_channels, kernel_size=1)
        self.bn_project = nn.BatchNorm1d(out_channels)
        self.silu = nn.SiLU()

        self.needs_proj = in_channels != out_channels
        self.res_proj = nn.Conv1d(in_channels, out_channels, 1) if self.needs_proj else nn.Identity()

    def forward(self, x, pad_mask_ct=None):
        if pad_mask_ct is not None: x = x.masked_fill(pad_mask_ct, 0.0)
        res = self.res_proj(x)

        x = self.silu(self.bn_expand(self.expand(x)))
        if pad_mask_ct is not None: x = x.masked_fill(pad_mask_ct, 0.0)

        x = self.silu(self.bn_depthwise(self.depthwise(x)))
        x = self.se(x)
        x = self.bn_project(self.project(x))

        x = res + x
        if pad_mask_ct is not None: x = x.masked_fill(pad_mask_ct, 0.0)
        return x

# ============================================================================
# 4. CUSTOM CONFORMER BLOCK
# ============================================================================
class FeedForwardModule(nn.Module):
    def __init__(self, d_model, expansion=4, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, d_model * expansion),
            nn.SiLU(),
            nn.Dropout(dropout),
            nn.Linear(d_model * expansion, d_model),
            nn.Dropout(dropout)
        )
    def forward(self, x):
        return self.net(x)

class ConformerConvModule(nn.Module):
    def __init__(self, d_model, kernel_size=15, dropout=0.1):
        super().__init__()
        self.ln = nn.LayerNorm(d_model)
        self.pw1 = nn.Conv1d(d_model, d_model * 2, 1)  # *2 for GLU
        self.dw = nn.Conv1d(d_model, d_model, kernel_size, padding=kernel_size // 2, groups=d_model)
        self.bn = nn.BatchNorm1d(d_model)
        self.silu = nn.SiLU()
        self.pw2 = nn.Conv1d(d_model, d_model, 1)
        self.drop = nn.Dropout(dropout)

    def forward(self, x, pad_mask_ct=None):
        # x is (B, T, D) -> needs (B, D, T) for Conv1d
        x = self.ln(x).transpose(1, 2)
        if pad_mask_ct is not None: x = x.masked_fill(pad_mask_ct, 0.0)

        x = self.pw1(x)
        x = F.glu(x, dim=1)

        x = self.dw(x)
        x = self.bn(x)
        x = self.silu(x)

        x = self.pw2(x)
        x = self.drop(x)

        x = x.transpose(1, 2)  # back to (B, T, D)
        if pad_mask_ct is not None:
            x = x.masked_fill(pad_mask_ct.transpose(1, 2), 0.0)
        return x

class ConformerBlock(nn.Module):
    def __init__(self, d_model, num_heads, conv_kernel_size=15, dropout=0.1):
        super().__init__()
        self.ffn1 = FeedForwardModule(d_model, dropout=dropout)
        self.attn_ln = nn.LayerNorm(d_model)
        self.attn = nn.MultiheadAttention(d_model, num_heads, dropout=dropout, batch_first=True)
        self.conv = ConformerConvModule(d_model, conv_kernel_size, dropout)
        self.ffn2 = FeedForwardModule(d_model, dropout=dropout)
        self.final_ln = nn.LayerNorm(d_model)

    def forward(self, x, pad_mask_bt, pad_mask_ct):
        x = x + 0.5 * self.ffn1(x)

        res = x
        x = self.attn_ln(x)
        x, _ = self.attn(x, x, x, key_padding_mask=pad_mask_bt)
        x = res + x

        x = x + self.conv(x, pad_mask_ct)
        x = x + 0.5 * self.ffn2(x)

        return self.final_ln(x)

# ============================================================================
# 5. THE MAIN ARCHITECTURE
# ============================================================================
class ISL_Conformer(nn.Module):
    def __init__(
        self,
        input_dim=163,
        num_classes=100,
        d_model=256,
        num_heads=4,
        num_layers=4,
        conv_kernel_size=15,
        dropout=0.1
    ):
        super().__init__()
        self.input_stage = MultiStreamEmbedding(d_model=d_model)

        self.transpose_to_ct = Transpose(1, 2)
        self.mbconv = EfficientNet1DBlock(in_channels=d_model, out_channels=d_model)
        self.pool = nn.MaxPool1d(kernel_size=2)
        self.transpose_to_tc = Transpose(1, 2)

        self.conformer_layers = nn.ModuleList([
            ConformerBlock(d_model, num_heads, conv_kernel_size, dropout)
            for _ in range(num_layers)
        ])

        self.classifier = nn.Linear(d_model, num_classes + 1)  # +1 for CTC blank

    def forward(self, x, lengths):
        B, T, _ = x.shape
        pad_mask_bt = lengths_to_padding_mask(lengths, T)
        pad_mask_ct = pad_mask_bt.unsqueeze(1)

        x = self.input_stage(x)

        x = self.transpose_to_ct(x)
        x = self.mbconv(x, pad_mask_ct)
        x = self.pool(x)

        lengths_pooled = lengths // 2
        T_pooled = x.size(-1)
        pad_mask_bt_pooled = lengths_to_padding_mask(lengths_pooled, T_pooled)
        pad_mask_ct_pooled = pad_mask_bt_pooled.unsqueeze(1)

        mbconv_features = self.transpose_to_tc(x)  # (B, T_pooled, D)

        x_conf = mbconv_features
        for layer in self.conformer_layers:
            x_conf = layer(x_conf, pad_mask_bt_pooled, pad_mask_ct_pooled)

        x_fused = x_conf + mbconv_features

        out = self.classifier(x_fused)
        out = out.masked_fill(pad_mask_bt_pooled.unsqueeze(-1), 0.0)

        return F.log_softmax(out, dim=-1), lengths_pooled

# ============================================================================
# SMOKE TEST
# ============================================================================
_model = ISL_Conformer(num_classes=100)
_dummy_x = torch.randn(2, 60, 163)
_dummy_lengths = torch.tensor([60, 45])
_log_probs, _pooled_lengths = _model(_dummy_x, _dummy_lengths)
print("✅ Model smoke test passed.")
print(f"Input Shape: {_dummy_x.shape} -> Output Shape: {_log_probs.shape}, Pooled Lengths: {_pooled_lengths.tolist()}")
del _model, _dummy_x, _dummy_lengths, _log_probs, _pooled_lengths


## 4. Shared feature extractor (SAME schema for word images / sentence frames / video)
This is the fix for the biggest bug in the original notebook: 3 different extraction schemas across stages meant the word-level pretraining never actually connected to the video-trained model. Now there is exactly **one** function, `extract_frame_features()`, called identically everywhere. Output is always 163-dim: `lh(63) + rh(63) + arms(12) + face(21) + flags(4)`.

In [ ]:
import numpy as np
import torch

def extract_frame_features(hands_result, pose_result, face_result):
    """Canonical 163-dim feature extraction, shared by ALL stages (word image / sentence frame / video frame).

    Layout (must match MultiStreamEmbedding slicing exactly):
      [0:63]    lh (21 landmarks x 3)
      [63:126]  rh (21 landmarks x 3)
      [126:138] arms (4 landmarks x 3)  -> shoulders(11,12) + elbows(13,14)
      [138:159] face (7 landmarks x 3)  -> eyes/nose/cheeks/mouth corners
      [159:163] flags [lh_flag, rh_flag, arms_flag, face_flag]
    """
    # --- Hands ---
    lh = np.zeros((21, 3)); rh = np.zeros((21, 3))
    lh_flag = 0.0; rh_flag = 0.0
    if hands_result and hands_result.hand_landmarks:
        for idx, handedness in enumerate(hands_result.handedness):
            label = handedness[0].category_name
            pts = np.array([[lm.x, lm.y, lm.z] for lm in hands_result.hand_landmarks[idx]])
            if label == 'Left':
                lh = pts; lh_flag = 1.0
            else:
                rh = pts; rh_flag = 1.0

    # --- Arms (shoulders + elbows) ---
    arms = np.zeros((4, 3))
    arms_flag = 0.0
    if pose_result and pose_result.pose_landmarks:
        p = pose_result.pose_landmarks[0]
        arms = np.array([
            [p[11].x, p[11].y, p[11].z], [p[12].x, p[12].y, p[12].z],
            [p[13].x, p[13].y, p[13].z], [p[14].x, p[14].y, p[14].z],
        ])
        arms_flag = 1.0

    # --- Face (7 pts: eyes, nose tip, cheeks, mouth corners) ---
    face_pts = np.zeros((7, 3))
    face_flag = 0.0
    if face_result and face_result.face_landmarks:
        f = face_result.face_landmarks[0]
        face_pts = np.array([
            [f[159].x, f[159].y, f[159].z], [f[386].x, f[386].y, f[386].z],
            [f[4].x, f[4].y, f[4].z],
            [f[234].x, f[234].y, f[234].z], [f[454].x, f[454].y, f[454].z],
            [f[61].x, f[61].y, f[61].z],    [f[291].x, f[291].y, f[291].z],
        ])
        face_flag = 1.0

    # --- Normalization: anchor to nose, scale by shoulder width (PER-FRAME, not sequence-mean) ---
    anchor = face_pts[2] if face_flag else np.array([0.0, 0.0, 0.0])
    if lh_flag: lh = lh - anchor
    if rh_flag: rh = rh - anchor
    if arms_flag: arms = arms - anchor
    if face_flag: face_pts = face_pts - anchor

    scale = np.linalg.norm(arms[0] - arms[1]) if arms_flag else 0.0
    if scale > 0.05:
        lh = lh / scale; rh = rh / scale; arms = arms / scale; face_pts = face_pts / scale

    frame_data = np.concatenate([
        lh.flatten(), rh.flatten(), arms.flatten(), face_pts.flatten(),
        [lh_flag, rh_flag, arms_flag, face_flag]
    ])
    return frame_data.astype(np.float32)  # (163,)


def extract_frame_features_video_mode(hands_lh, hands_rh, hands_lh_flag, hands_rh_flag,
                                       pose_landmarks, face_landmarks,
                                       last_lh, last_rh, last_arms, last_face):
    """Variant for VIDEO streams: carries forward last-known landmarks on tracking dropout
    instead of zeroing (prevents 'teleporting' hands / spurious zero-frames that corrupt
    per-frame normalization). Returns (frame_features, new_last_lh, new_last_rh, new_last_arms, new_last_face).
    """
    lh = hands_lh if hands_lh_flag else last_lh
    rh = hands_rh if hands_rh_flag else last_rh
    lh_flag = 1.0 if hands_lh_flag else (1.0 if np.any(last_lh != 0) else 0.0)
    rh_flag = 1.0 if hands_rh_flag else (1.0 if np.any(last_rh != 0) else 0.0)

    if pose_landmarks:
        p = pose_landmarks[0] if isinstance(pose_landmarks[0], list) else pose_landmarks
        arms = np.array([[p[11].x, p[11].y, p[11].z], [p[12].x, p[12].y, p[12].z],
                          [p[13].x, p[13].y, p[13].z], [p[14].x, p[14].y, p[14].z]])
        arms_flag = 1.0
    else:
        arms = last_arms
        arms_flag = 1.0 if np.any(last_arms != 0) else 0.0

    if face_landmarks:
        f = face_landmarks[0] if isinstance(face_landmarks[0], list) else face_landmarks
        face_pts = np.array([
            [f[159].x, f[159].y, f[159].z], [f[386].x, f[386].y, f[386].z],
            [f[4].x, f[4].y, f[4].z],
            [f[234].x, f[234].y, f[234].z], [f[454].x, f[454].y, f[454].z],
            [f[61].x, f[61].y, f[61].z],    [f[291].x, f[291].y, f[291].z],
        ])
        face_flag = 1.0
    else:
        face_pts = last_face
        face_flag = 1.0 if np.any(last_face != 0) else 0.0

    anchor = face_pts[2] if face_flag else np.array([0.0, 0.0, 0.0])
    lh2 = lh - anchor if lh_flag else lh
    rh2 = rh - anchor if rh_flag else rh
    arms2 = arms - anchor if arms_flag else arms
    face2 = face_pts - anchor if face_flag else face_pts

    scale = np.linalg.norm(arms2[0] - arms2[1]) if arms_flag else 0.0
    if scale > 0.05:
        lh2 = lh2 / scale; rh2 = rh2 / scale; arms2 = arms2 / scale; face2 = face2 / scale

    frame_data = np.concatenate([
        lh2.flatten(), rh2.flatten(), arms2.flatten(), face2.flatten(),
        [lh_flag, rh_flag, arms_flag, face_flag]
    ]).astype(np.float32)

    return frame_data, lh, rh, arms, face_pts

print("✅ Shared extraction functions defined (163-dim schema, used by all 3 stages).")


## 5. STAGE 1 — Word-level image extraction (163-dim, canonical schema)

In [ ]:
import os
import cv2
import glob
import torch
import numpy as np
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
from tqdm.notebook import tqdm

WORD_LEVEL_DIR = "/content/isl_csltr_dataset/ISL_CSLRT_Corpus/ISL_CSLRT_Corpus/Frames_Word_Level"
OUTPUT_DIR_WORD = "/content/tensors_word_level_163"
os.makedirs(OUTPUT_DIR_WORD, exist_ok=True)

BaseOptions = mp.tasks.BaseOptions
VisionMode = mp.tasks.vision.RunningMode.IMAGE

hand_options = vision.HandLandmarkerOptions(base_options=BaseOptions(model_asset_path="hand_landmarker.task"),
                                             running_mode=VisionMode, num_hands=2)
pose_options = vision.PoseLandmarkerOptions(base_options=BaseOptions(model_asset_path="pose_landmarker.task"),
                                             running_mode=VisionMode)
face_options = vision.FaceLandmarkerOptions(base_options=BaseOptions(model_asset_path="face_landmarker.task"),
                                             running_mode=VisionMode)

def process_static_image(img_path, hand_lm, pose_lm, face_lm):
    frame = cv2.imread(img_path)
    if frame is None:
        return None
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=frame_rgb)

    hands = hand_lm.detect(mp_image)
    pose = pose_lm.detect(mp_image)
    face = face_lm.detect(mp_image)

    feats = extract_frame_features(hands, pose, face)  # (163,) — shared function
    return torch.tensor(np.expand_dims(feats, axis=0), dtype=torch.float32)  # (1, 163)

print(f"🔍 Scanning {WORD_LEVEL_DIR} for images...")
all_images = []
for ext in ('*.jpg', '*.jpeg', '*.png', '*.JPG', '*.PNG'):
    all_images.extend(glob.glob(f"{WORD_LEVEL_DIR}/**/{ext}", recursive=True))

print(f"⚙️ Found {len(all_images)} images. Starting extraction...")
success_count = 0

with vision.HandLandmarker.create_from_options(hand_options) as hand_lm, \
     vision.PoseLandmarker.create_from_options(pose_options) as pose_lm, \
     vision.FaceLandmarker.create_from_options(face_options) as face_lm:

    for img_path in tqdm(all_images):
        class_name = os.path.basename(os.path.dirname(img_path))
        img_name = os.path.splitext(os.path.basename(img_path))[0]
        save_dir = os.path.join(OUTPUT_DIR_WORD, class_name)
        os.makedirs(save_dir, exist_ok=True)
        save_path = os.path.join(save_dir, f"{img_name}.pt")

        tensor_data = process_static_image(img_path, hand_lm, pose_lm, face_lm)
        if tensor_data is not None:
            torch.save(tensor_data, save_path)
            success_count += 1

print(f"✅ Word-level extraction complete! {success_count}/{len(all_images)} images processed.")
print("   -> Saved as 163-dim tensors, SAME schema as sentence/video stages below.")


### Backup word-level tensors to Drive

In [ ]:
import os, shutil
from google.colab import drive

drive.mount('/content/drive')
DRIVE_DIR_WORD = "/content/drive/MyDrive/ISL_Dataset/Word_Level_Tensors_163"
os.makedirs(DRIVE_DIR_WORD, exist_ok=True)
print("⏳ Copying word-level tensors to Google Drive...")
shutil.copytree(OUTPUT_DIR_WORD, DRIVE_DIR_WORD, dirs_exist_ok=True)
print("✅ Backup complete.")


## 6. STAGE 2 — Sentence-level frame extraction (163-dim, same schema)
Sentence-level samples are frame sequences (not single images) whose target is a **sequence of word indices**, built from the corpus's sentence→gloss mapping. Adjust `SENTENCE_FRAMES_DIR` / the gloss-lookup logic to match how the ISL-CSLTR sentence corpus stores its word-level annotation for each sentence video (the Kaggle dataset ships a `corpus.csv`/annotation file — check its exact columns, this cell assumes a `sentence, words` style mapping and will need light adjustment).

In [ ]:
import os, glob, cv2, torch, numpy as np
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
from tqdm.notebook import tqdm

# NOTE: actual structure confirmed from the dataset is TWO levels deep:
#   Frames_Sentence_Level/<sentence phrase>/<repetition 1..7>/*.jpg
# Each repetition folder is one ordered take of that sentence by a signer.
# We save ONE tensor per repetition (not one per phrase), since each repetition
# is an independent sequence sample for that sentence class.
SENTENCE_FRAMES_DIR = "/content/isl_csltr_dataset/ISL_CSLRT_Corpus/ISL_CSLRT_Corpus/Frames_Sentence_Level"
OUTPUT_DIR_SENTENCE = "/content/tensors_sentence_level_163"
os.makedirs(OUTPUT_DIR_SENTENCE, exist_ok=True)

def process_sentence_repetition_folder(folder_path, hand_lm, pose_lm, face_lm):
    frame_paths = sorted(glob.glob(os.path.join(folder_path, "*.jpg")) +
                          glob.glob(os.path.join(folder_path, "*.png")))
    if not frame_paths:
        return None

    seq = []
    for fp in frame_paths:
        frame = cv2.imread(fp)
        if frame is None:
            continue
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=frame_rgb)

        hands = hand_lm.detect(mp_image)
        pose = pose_lm.detect(mp_image)
        face = face_lm.detect(mp_image)
        feats = extract_frame_features(hands, pose, face)  # (163,) same shared fn
        seq.append(feats)

    if not seq:
        return None
    return torch.tensor(np.array(seq), dtype=torch.float32)  # (T, 163)

if os.path.exists(SENTENCE_FRAMES_DIR):
    phrase_dirs = [d for d in glob.glob(f"{SENTENCE_FRAMES_DIR}/*") if os.path.isdir(d)]

    # Build the flat list of (phrase, repetition_folder) pairs first, so tqdm shows real progress
    rep_folders = []
    for phrase_dir in phrase_dirs:
        phrase_name = os.path.basename(phrase_dir)
        for rep_dir in glob.glob(f"{phrase_dir}/*"):
            if os.path.isdir(rep_dir):
                rep_folders.append((phrase_name, rep_dir))

    print(f"⚙️ Found {len(phrase_dirs)} sentence phrases, {len(rep_folders)} total repetition folders. Extracting...")
    success_count = 0

    with vision.HandLandmarker.create_from_options(hand_options) as hand_lm, \
         vision.PoseLandmarker.create_from_options(pose_options) as pose_lm, \
         vision.FaceLandmarker.create_from_options(face_options) as face_lm:

        for phrase_name, rep_dir in tqdm(rep_folders):
            rep_id = os.path.basename(rep_dir)
            save_dir = os.path.join(OUTPUT_DIR_SENTENCE, phrase_name)
            os.makedirs(save_dir, exist_ok=True)
            save_path = os.path.join(save_dir, f"{rep_id}.pt")

            tensor_data = process_sentence_repetition_folder(rep_dir, hand_lm, pose_lm, face_lm)
            if tensor_data is not None:
                torch.save(tensor_data, save_path)
                success_count += 1

    print(f"✅ Sentence-level extraction complete! {success_count}/{len(rep_folders)} repetitions processed.")
    print(f"   -> Saved as: {OUTPUT_DIR_SENTENCE}/<phrase>/<repetition_id>.pt")
else:
    print(f"⚠️ {SENTENCE_FRAMES_DIR} not found — check the dataset path.")


### Gloss / word-sequence targets for sentence-level CTC

The dataset ships a real gloss annotation file: `ISL Corpus sign glosses.csv` (columns `Sentence`, `SIGN GLOSSES`), e.g. `"are you free today"` -> `"YOU FREE TODAY"`. We use this directly instead of guessing from the sentence text.

However, of the 170 unique gloss tokens, only 88 have an exact matching folder under `Frames_Word_Level/`. The other 82 fall into 3 buckets, handled below:

1. **Alias / merged-folder tokens** (e.g. `HIDE` -> folder `HIDING`, `I`/`ME`/`MY` -> folder `I_ME_MINE_MY`, `TAKE`+`CARE` -> folder `TAKE CARE`) — mapped via an explicit alias table.
2. **CSV data-entry artifacts** (`(AGE)`, `XXXXXXXX`, trailing-punctuation tokens like `FINE.`, `MEDICINE,`, typos like `CONGRATULATIIONS`) — cleaned/corrected via the same alias table.
3. **Genuine function words with no isolated sign in this corpus** (`A, THE, TO, IT, HE, SHE, WE, OF, IN, ON, WITH, BY, ...`) — **dropped from the gloss sequence** for that sentence (not the whole sentence). This mirrors how the corpus's own `SIGN GLOSSES` column already drops words like "are" from "are you free today", so it's consistent with the annotation's own convention, not an inconsistency we're introducing.

Every drop is logged so you can audit coverage and extend the alias table later — nothing is silently guessed.

In [ ]:
import os
import pandas as pd
from collections import Counter

GLOSS_CSV = "/content/isl_csltr_dataset/ISL_CSLRT_Corpus/ISL_CSLRT_Corpus/corpus_csv_files/ISL Corpus sign glosses.csv"
WORD_FRAMES_DIR = "/content/isl_csltr_dataset/ISL_CSLRT_Corpus/ISL_CSLRT_Corpus/Frames_Word_Level"

# --- 1. Alias table: gloss token -> real Frames_Word_Level folder name (or a SPACE-separated
#     sequence of real folder names, for tokens that expand to multiple signs) ---
# Built from comparing the 170 unique gloss tokens against the 88 matching folders (see chat).
GLOSS_ALIASES = {
    # merged / compound folders (a folder covers 2+ gloss tokens signed together)
    "HI": "HELLO_HI",
    "I": "I_ME_MINE_MY", "ME": "I_ME_MINE_MY", "MY": "I_ME_MINE_MY",
    "LOVE": "LIKE_LOVE",
    "COLLEGE": "COLLEGE_SCHOOL", "SCHOOL": "COLLEGE_SCHOOL",
    "DONOT": "NOT", "DONT": "NOT",
    "OLD": "OLD_AGE",
    "SOMEONE": "SOME ONE",
    # NOTE: "TAKE" + "CARE" as separate tokens both map to standalone folders if they exist
    # alone elsewhere; but when they appear ADJACENT in a gloss sequence we prefer the merged
    # folder. Handled as a bigram rule below, not a simple 1:1 dict entry.

    # spelling/typo/tense variants
    "HIDE": "HIDING",
    "CRY": "CRYING",
    "STOPPED": "STOP",
    "ENJOYED": "ENJOY",
    "COMING": "COME",
    "CONGRATULATIIONS": "CONGRATULATIONS",

    # punctuation/artifact cleanup -> map to the clean token, re-resolved recursively
    "ANYTHING,": "ANYTHING",
    "FINE.": "FINE",
    "MEDICINE,": "MEDICINE",
}

# Tokens with a real, adjacent-pair merge into ONE folder (checked as bigrams before unigram lookup)
GLOSS_BIGRAM_ALIASES = {
    ("TAKE", "CARE"): "TAKE CARE",
    ("TAKE", "TIME"): "TAKE TIME",
    ("DON'T", "CARE"): "DON'T CARE",
    ("DONT", "CARE"): "DON'T CARE",
}

# Tokens confirmed as CSV artifacts or function words with NO isolated sign in this corpus.
# These are DROPPED from the gloss sequence (not the whole sentence).
GLOSS_DROP_TOKENS = {
    "(AGE)", "XXXXXXXX",
    "A", "ABOUT", "AM", "ANY", "BE", "BY", "CAME", "CAREER", "GLASS", "GOT", "HAIR",
    "HAVE", "HE", "HIM", "IN", "INTO", "IT", "KNOW", "LET", "LIGHT", "LOT", "MAKE",
    "MEAN", "MORE", "MUCH", "NEED", "NEVER", "NO", "NOW", "OF", "OFF", "ON", "ONE",
    "ONWARDS", "PLAN", "SHE", "SIR", "SO", "SOME", "SOMEHOW", "STOPPED_DUPLICATE",
    "SUFFERING", "THE", "THERE", "THIS", "TIME", "TO", "TRY", "TURN", "VERY", "WAY",
    "WE", "WHEN", "WHICH", "WHY", "WITH", "YOUR", "YOURSELF",
}
# NOTE: "TIME" is in both GLOSS_BIGRAM_ALIASES (as part of "TAKE TIME") and GLOSS_DROP_TOKENS
# (as a standalone word elsewhere, e.g. "now onwards..."). The bigram check runs FIRST, so
# "TAKE TIME" resolves correctly; a standalone "TIME" not preceded by "TAKE" gets dropped.
# Same logic applies to "CARE" (kept as bigram with TAKE/DON'T, dropped standalone... though
# "CARE" alone isn't in GLOSS_DROP_TOKENS above since it wasn't in the missing-tokens list —
# it only ever appeared as part of a bigram in this corpus).

word_folders = set(os.listdir(WORD_FRAMES_DIR))

def resolve_gloss_sequence(gloss_string, log_drops=None):
    """Convert a raw 'SIGN GLOSSES' string into a list of real Frames_Word_Level folder names.
    Applies bigram aliases first, then unigram aliases, then drops confirmed function words/artifacts.
    Any token that's still unresolved (genuinely unknown) causes the WHOLE sentence to be skipped
    (logged), since an unrecognized token is a sign we can't confidently confirm is safe to drop.
    """
    tokens = gloss_string.strip().split()
    resolved = []
    i = 0
    unknown = []
    while i < len(tokens):
        # Try bigram merge first
        if i + 1 < len(tokens) and (tokens[i], tokens[i + 1]) in GLOSS_BIGRAM_ALIASES:
            folder = GLOSS_BIGRAM_ALIASES[(tokens[i], tokens[i + 1])]
            resolved.append(folder)
            i += 2
            continue

        tok = tokens[i]
        if tok in word_folders:
            resolved.append(tok)
        elif tok in GLOSS_ALIASES:
            aliased = GLOSS_ALIASES[tok]
            if aliased in word_folders:
                resolved.append(aliased)
            else:
                unknown.append(tok)  # alias target itself doesn't exist — flag it
        elif tok in GLOSS_DROP_TOKENS:
            if log_drops is not None:
                log_drops.append(tok)
        else:
            unknown.append(tok)
        i += 1

    return resolved, unknown

# --- 2. Build sentence_to_words from the CSV ---
sentence_to_words = {}
drop_log = []
skipped_sentences = []

if os.path.exists(GLOSS_CSV):
    df = pd.read_csv(GLOSS_CSV)
    for _, row in df.iterrows():
        sentence = row["Sentence"]
        gloss_str = row["SIGN GLOSSES"]
        if pd.isna(sentence) or pd.isna(gloss_str):
            continue

        this_drops = []
        resolved, unknown = resolve_gloss_sequence(gloss_str, log_drops=this_drops)

        if unknown:
            skipped_sentences.append((sentence, gloss_str, unknown))
            continue  # unresolved token -> skip this sentence entirely, don't guess

        if not resolved:
            skipped_sentences.append((sentence, gloss_str, ["<all tokens dropped>"]))
            continue

        sentence_to_words[sentence] = resolved
        drop_log.extend(this_drops)

    print(f"✅ Resolved gloss sequences for {len(sentence_to_words)}/{len(df)} sentences.")
    print(f"⚠️ Skipped {len(skipped_sentences)} sentences due to unresolved tokens:")
    for sentence, gloss_str, unknown in skipped_sentences[:15]:
        print(f"   '{sentence}' (gloss: '{gloss_str}') -> unknown: {unknown}")
    if len(skipped_sentences) > 15:
        print(f"   ... and {len(skipped_sentences) - 15} more")

    drop_counts = Counter(drop_log)
    print(f"\n📊 Function-word tokens dropped (kept sentence, removed token) — top 15 by frequency:")
    for tok, count in drop_counts.most_common(15):
        print(f"   {tok}: {count}")
else:
    print(f"❌ {GLOSS_CSV} not found — check the exact filename/path in your dataset.")


## 7. STAGE 3 — Video extraction (163-dim, same schema, VIDEO running mode + last-known carry-forward)

In [ ]:
import os, cv2, glob, shutil, torch, numpy as np
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
from tqdm.notebook import tqdm

RAW_DATA_DIR = "/content/isl_csltr_dataset"
OUTPUT_DIR_VIDEO = "/content/tensors_video_163"
os.makedirs(OUTPUT_DIR_VIDEO, exist_ok=True)

task_path = "holistic_landmarker.task"
if not os.path.exists(task_path):
    print("📥 Downloading MediaPipe Holistic model...")
    import urllib.request
    urllib.request.urlretrieve(
        "https://storage.googleapis.com/mediapipe-models/holistic_landmarker/holistic_landmarker/float16/latest/holistic_landmarker.task",
        task_path
    )

video_options = vision.HolisticLandmarkerOptions(
    base_options=python.BaseOptions(model_asset_path=task_path),
    running_mode=vision.RunningMode.VIDEO
)

def process_video(video_path, landmarker):
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    sequence = []
    frame_index = 0

    last_lh = np.zeros((21, 3)); last_rh = np.zeros((21, 3))
    last_arms = np.zeros((4, 3)); last_face = np.zeros((7, 3))

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=frame_rgb)
        timestamp_ms = int((frame_index * 1000) / fps)
        frame_index += 1

        results = landmarker.detect_for_video(mp_image, timestamp_ms)

        lh_raw = np.array([[lm.x, lm.y, lm.z] for lm in results.left_hand_landmarks[0]]) \
                 if results.left_hand_landmarks else np.zeros((21, 3))
        rh_raw = np.array([[lm.x, lm.y, lm.z] for lm in results.right_hand_landmarks[0]]) \
                 if results.right_hand_landmarks else np.zeros((21, 3))
        lh_flag = 1.0 if results.left_hand_landmarks else 0.0
        rh_flag = 1.0 if results.right_hand_landmarks else 0.0

        pose_landmarks = results.pose_landmarks[0] if results.pose_landmarks else None
        face_landmarks = results.face_landmarks[0] if results.face_landmarks else None

        feats, last_lh, last_rh, last_arms, last_face = extract_frame_features_video_mode(
            lh_raw, rh_raw, lh_flag, rh_flag, pose_landmarks, face_landmarks,
            last_lh, last_rh, last_arms, last_face
        )
        sequence.append(feats)

    cap.release()
    if not sequence:
        return None
    return torch.tensor(np.array(sequence), dtype=torch.float32)  # (T, 163)

print("⚙️ Extracting video tensors (163-dim, canonical schema)...")
video_files = glob.glob(f"{RAW_DATA_DIR}/**/*.mp4", recursive=True)
success_count = 0

for vid_path in tqdm(video_files):
    relative_path = os.path.relpath(vid_path, RAW_DATA_DIR)
    save_dir = os.path.join(OUTPUT_DIR_VIDEO, os.path.dirname(relative_path))
    os.makedirs(save_dir, exist_ok=True)
    save_path = os.path.join(save_dir, os.path.basename(vid_path).replace(".mp4", ".pt"))

    # Re-init per video: resets timestamps, avoids smoothing bleed-over between clips
    with vision.HolisticLandmarker.create_from_options(video_options) as landmarker:
        tensor_data = process_video(vid_path, landmarker)
        if tensor_data is not None:
            torch.save(tensor_data, save_path)
            success_count += 1

print(f"✅ Video extraction complete! {success_count}/{len(video_files)} videos processed.")


### Backup video tensors to Drive

In [ ]:
import os, shutil
from google.colab import drive

drive.mount('/content/drive')
DRIVE_SAVE_DIR = "/content/drive/MyDrive/ISL_Project"
os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)
DRIVE_ZIP_PATH = os.path.join(DRIVE_SAVE_DIR, "tensors_video_163_backup.zip")

print("🗜️ Compressing dataset...")
shutil.make_archive("/content/tensors_video_163_backup", 'zip', OUTPUT_DIR_VIDEO)
shutil.copy("/content/tensors_video_163_backup.zip", DRIVE_ZIP_PATH)
print(f"✅ Backed up at {DRIVE_ZIP_PATH}")


### Reload video tensors from Drive (run this instead of re-extracting on a fresh runtime)

In [ ]:
import os, zipfile
from google.colab import drive

drive.mount('/content/drive')
DRIVE_ZIP_PATH = "/content/drive/MyDrive/ISL_Project/tensors_video_163_backup.zip"
LOCAL_EXTRACT_DIR = "/content/tensors_video_163"

if os.path.exists(DRIVE_ZIP_PATH):
    os.makedirs(LOCAL_EXTRACT_DIR, exist_ok=True)
    with zipfile.ZipFile(DRIVE_ZIP_PATH, 'r') as zip_ref:
        zip_ref.extractall(LOCAL_EXTRACT_DIR)
    print(f"✅ Loaded into {LOCAL_EXTRACT_DIR}")
else:
    print(f"❌ {DRIVE_ZIP_PATH} not found.")


## 8. EDA — raw extracted tensors (word-level)

In [ ]:
import os, glob, torch, random
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

DATA_DIR = OUTPUT_DIR_WORD
all_files = glob.glob(f"{DATA_DIR}/**/*.pt", recursive=True)
print(f"🔍 Analyzing {len(all_files)} word-level tensors...\n")

classes = sorted(set(os.path.basename(os.path.dirname(p)) for p in all_files))
print(f"📊 {len(classes)} word classes found.")

from collections import Counter
label_counts = Counter(os.path.basename(os.path.dirname(p)) for p in all_files)
words, counts = zip(*label_counts.most_common())
plt.figure(figsize=(16, 4))
plt.bar(words, counts, color='mediumseagreen', edgecolor='black')
plt.title("Samples per Word Class (raw, before balancing)")
plt.xticks(rotation=90, fontsize=7)
plt.ylabel("Count")
plt.tight_layout()
plt.show()

def plot_163_skeleton(vec):
    """Unpack a 163-dim frame vector into lh/rh/arms/face for visualization."""
    lh = vec[0:63].reshape(21, 3)
    rh = vec[63:126].reshape(21, 3)
    arms = vec[126:138].reshape(4, 3)
    face = vec[138:159].reshape(7, 3)
    flags = vec[159:163]

    fig = plt.figure(figsize=(7, 7))
    ax = fig.add_subplot(111, projection='3d')
    def plot_group(pts, color, label, flag):
        if flag:
            ax.scatter(pts[:, 0], -pts[:, 1], -pts[:, 2], c=color, label=label, s=25)
    plot_group(lh, 'green', 'Left Hand', flags[0])
    plot_group(rh, 'blue', 'Right Hand', flags[1])
    plot_group(arms, 'gray', 'Arms', flags[2])
    plot_group(face, 'red', 'Face', flags[3])
    ax.set_title("163-dim Frame Reconstruction")
    ax.legend()
    plt.show()

if all_files:
    sample_file = random.choice(all_files)
    sample_tensor = torch.load(sample_file)
    print(f"🦴 Visualizing: {sample_file}")
    plot_163_skeleton(sample_tensor[0].numpy())


## 9. Augmentation + split-then-balance dataset (word-level)
**Fix vs. original**: split real (non-synthetic) samples into train/val **first**, grouped so a source sample's synthetic children always stay on the same side, *then* oversample only inside train. This removes the synthetic-sibling leakage across train/val that existed in the original pipeline.

In [ ]:
import os, glob, math, random, copy
import torch
import numpy as np
from collections import defaultdict
from torch.utils.data import Dataset, DataLoader, Subset

# --- Spatial augmentation (word-level: single frame, shape (1,163)) ---
class SaneSkeletalAugmentation:
    def __init__(self, max_rot_degrees=10.0, max_shift=0.05, noise_std=0.01, apply_prob=0.7):
        self.max_rot_rad = math.radians(max_rot_degrees)
        self.max_shift = max_shift
        self.noise_std = noise_std
        self.apply_prob = apply_prob

    def __call__(self, tensor):
        if torch.rand(1).item() > self.apply_prob:
            return tensor
        aug = tensor.clone()
        coords = aug[0, :159].view(53, 3)  # 53 pts = lh21+rh21+arms4+face7

        angle = (torch.rand(1).item() * 2 - 1) * self.max_rot_rad
        cos_a, sin_a = math.cos(angle), math.sin(angle)
        x, y = coords[:, 0].clone(), coords[:, 1].clone()
        coords[:, 0] = x * cos_a - y * sin_a
        coords[:, 1] = x * sin_a + y * cos_a

        coords[:, 0] += (torch.rand(1).item() * 2 - 1) * self.max_shift
        coords[:, 1] += (torch.rand(1).item() * 2 - 1) * self.max_shift
        coords += torch.randn_like(coords) * self.noise_std

        aug[0, :159] = coords.view(159)
        return aug

WORD_DATA_DIR = OUTPUT_DIR_WORD

class ISLWordLevelDataset(Dataset):
    """Word-level dataset built from a pre-split, pre-balanced file list (see splitting cell below)."""
    def __init__(self, file_label_pairs, class_to_idx, transform=None):
        self.pairs = file_label_pairs
        self.class_to_idx = class_to_idx
        self.transform = transform

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        path, label = self.pairs[idx]
        tensor = torch.load(path)  # (1, 163)
        if self.transform:
            tensor = self.transform(tensor)
        return tensor, torch.tensor(self.class_to_idx[label], dtype=torch.long)

# --- 1. Group real files by class, SPLIT FIRST (no synthetics exist yet at this point) ---
all_files = glob.glob(f"{WORD_DATA_DIR}/**/*.pt", recursive=True)
if not all_files:
    print(f"⚠️ No files found in {WORD_DATA_DIR} — run Stage 1 extraction first.")
else:
    classes = sorted(set(os.path.basename(os.path.dirname(p)) for p in all_files))
    class_to_idx = {c: i for i, c in enumerate(classes)}

    class_to_files = defaultdict(list)
    for p in all_files:
        class_to_files[os.path.basename(os.path.dirname(p))].append(p)

    train_pairs, val_pairs = [], []
    for cls, files in class_to_files.items():
        random.shuffle(files)
        split_point = max(1, int(0.8 * len(files)))
        for f in files[:split_point]:
            train_pairs.append((f, cls))
        for f in files[split_point:]:
            val_pairs.append((f, cls))

    print(f"✅ Real-data split: {len(train_pairs)} train / {len(val_pairs)} val "
          f"(no synthetic samples exist yet — split is leakage-free by construction)")

    # --- 2. Oversample ONLY the train split (per-class target count) ---
    TARGET_COUNT = 30
    train_by_class = defaultdict(list)
    for f, cls in train_pairs:
        train_by_class[cls].append(f)

    balanced_train_pairs = list(train_pairs)  # start with all real train samples
    SYN_DIR = "/content/tensors_word_level_163_synth"
    os.makedirs(SYN_DIR, exist_ok=True)

    for cls, files in train_by_class.items():
        current = len(files)
        i = 0
        while current < TARGET_COUNT and files:
            base_path = random.choice(files)
            base_tensor = torch.load(base_path)
            # simple synthetic: reuse SaneSkeletalAugmentation offline as a "hard copy" augmentation
            synth_aug = SaneSkeletalAugmentation(apply_prob=1.0)
            synth_tensor = synth_aug(base_tensor)
            synth_path = os.path.join(SYN_DIR, f"{cls}__synth_{i}.pt")
            torch.save(synth_tensor, synth_path)
            balanced_train_pairs.append((synth_path, cls))
            current += 1
            i += 1

    print(f"✅ Balanced train set: {len(balanced_train_pairs)} samples "
          f"(target {TARGET_COUNT}/class, synthetics only added to TRAIN, never VAL)")

    # --- 3. Datasets & loaders ---
    augmenter = SaneSkeletalAugmentation(max_rot_degrees=10.0, max_shift=0.05, noise_std=0.01)
    train_dataset = ISLWordLevelDataset(balanced_train_pairs, class_to_idx, transform=augmenter)
    val_dataset = ISLWordLevelDataset(val_pairs, class_to_idx, transform=None)

    train_loader_word = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2, pin_memory=True)
    val_loader_word = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

    num_classes_word = len(classes)
    print(f"✅ DataLoaders ready: {len(train_dataset)} train / {len(val_dataset)} val, "
          f"{num_classes_word} classes.")


## 10. EDA — balanced word-level dataset (post-augmentation)

In [ ]:
from collections import Counter
import matplotlib.pyplot as plt

if 'balanced_train_pairs' in locals():
    balanced_counts = Counter(cls for _, cls in balanced_train_pairs)
    words, counts = zip(*balanced_counts.most_common())
    plt.figure(figsize=(16, 4))
    plt.bar(words, counts, color='royalblue', edgecolor='black')
    plt.title("Balanced TRAIN Set: Samples per Word (should be ~flat at TARGET_COUNT)")
    plt.xticks(rotation=90, fontsize=7)
    plt.ylabel("Count")
    plt.tight_layout()
    plt.show()

    val_counts = Counter(cls for _, cls in val_pairs)
    print(f"📊 Val set class coverage: {len(val_counts)}/{len(classes)} classes have at least 1 val sample.")
    zero_val_classes = [c for c in classes if c not in val_counts]
    if zero_val_classes:
        print(f"⚠️ {len(zero_val_classes)} classes have ZERO validation samples "
              f"(too few real examples to split 80/20) — e.g. {zero_val_classes[:5]}")
else:
    print("⚠️ Run the previous cell first.")


## 11. Sentence-level Dataset (multi-word CTC targets) — NEW, did not exist before

In [ ]:
import os, glob, random, torch
from torch.utils.data import Dataset, DataLoader, Subset
from torch.nn.utils.rnn import pad_sequence

class ISLSentenceLevelDataset(Dataset):
    """Sentence-level dataset: each sample is a (T, 163) sequence with a MULTI-WORD target
    (list of word indices, in signing order) for CTC. Requires `sentence_to_words` (Section 6)
    and a shared `word_vocab` (built from the word-level class_to_idx, so word ids are consistent
    across stages for weight transfer).

    Files are laid out as: data_dir/<sentence phrase>/<repetition id>.pt
    (one tensor per repetition/take of that sentence — see Section 5 extraction).
    """

    def __init__(self, file_label_pairs, sentence_to_words, word_vocab):
        self.sentence_to_words = sentence_to_words
        self.word_vocab = word_vocab  # word -> idx (1-indexed, 0 reserved for CTC blank)

        self.samples = []
        for path, phrase in file_label_pairs:
            if phrase in self.sentence_to_words:
                words = self.sentence_to_words[phrase]
                if all(w in self.word_vocab for w in words):
                    self.samples.append((path, words))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, words = self.samples[idx]
        keypoints = torch.load(path)  # (T, 163)
        target = torch.tensor([self.word_vocab[w] for w in words], dtype=torch.long)
        return keypoints, target

def sentence_collate_fn(batch):
    inputs, targets = zip(*batch)
    in_lens = torch.tensor([x.shape[0] for x in inputs], dtype=torch.long)
    tgt_lens = torch.tensor([y.shape[0] for y in targets], dtype=torch.long)
    return (pad_sequence(inputs, batch_first=True, padding_value=0.0),
            pad_sequence(targets, batch_first=True, padding_value=0),
            in_lens, tgt_lens)

if 'sentence_to_words' in locals() and sentence_to_words and 'class_to_idx' in locals():
    word_vocab = {w: i + 1 for w, i in class_to_idx.items()}  # +1: 0 reserved for CTC blank

    # --- Recursive glob over the nested <phrase>/<rep_id>.pt layout ---
    all_sentence_files = glob.glob(f"{OUTPUT_DIR_SENTENCE}/*/*.pt")
    file_phrase_pairs = [(p, os.path.basename(os.path.dirname(p))) for p in all_sentence_files]

    # --- Split by PHRASE, not by individual repetition file ---
    # All repetitions of the same sentence phrase are highly similar (same signer/script,
    # different takes) — splitting at the repetition level would leak near-duplicate sequences
    # across train/val, same class of bug as the word-level oversampling leakage in Section 9.
    unique_phrases = sorted(set(phrase for _, phrase in file_phrase_pairs))
    random.shuffle(unique_phrases)
    split_point = max(1, int(0.8 * len(unique_phrases)))
    train_phrases = set(unique_phrases[:split_point])
    val_phrases = set(unique_phrases[split_point:])

    train_pairs = [(p, ph) for p, ph in file_phrase_pairs if ph in train_phrases]
    val_pairs = [(p, ph) for p, ph in file_phrase_pairs if ph in val_phrases]

    train_sentence_dataset = ISLSentenceLevelDataset(train_pairs, sentence_to_words, word_vocab)
    val_sentence_dataset = ISLSentenceLevelDataset(val_pairs, sentence_to_words, word_vocab)

    print(f"✅ Sentence-level dataset: {len(train_sentence_dataset)} train / {len(val_sentence_dataset)} val "
          f"samples, split by PHRASE ({len(train_phrases)}/{len(val_phrases)} phrases) — "
          f"no repetition of the same sentence appears on both sides.")

    if len(train_sentence_dataset) > 0 and len(val_sentence_dataset) > 0:
        train_loader_sentence = DataLoader(train_sentence_dataset, batch_size=8, shuffle=True,
                                            collate_fn=sentence_collate_fn, num_workers=2, pin_memory=True)
        val_loader_sentence = DataLoader(val_sentence_dataset, batch_size=8, shuffle=False,
                                          collate_fn=sentence_collate_fn, num_workers=2, pin_memory=True)
        print(f"✅ Sentence-level loaders ready.")
    else:
        print("⚠️ One of train/val is empty — check phrase count vs split ratio, or gloss resolution coverage.")
else:
    print("⚠️ sentence_to_words / class_to_idx not populated yet — run Section 6 (gloss resolution) "
          "and Section 9 (word-level split, builds class_to_idx) first.")


## 12. TRAIN — Stage 1 (word-level, `ISL_Conformer`, CTC with target length 1)

In [ ]:
import torch.optim as optim
from tqdm.notebook import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🚀 Training on {device}")

model = ISL_Conformer(input_dim=163, num_classes=num_classes_word).to(device)
criterion = torch.nn.CTCLoss(blank=0, zero_infinity=True)
optimizer = optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

def word_level_collate(batch):
    # batch items are (tensor(1,163), label). Wrap each single-frame sample as a length-1 "sequence"
    # so it flows through the same (B,T,163)+lengths interface ISL_Conformer expects.
    inputs = torch.stack([x for x, _ in batch])          # (B, 1, 163)
    labels = torch.stack([y for _, y in batch])          # (B,)
    lengths = torch.ones(len(batch), dtype=torch.long)   # each sample is 1 frame
    targets = labels.unsqueeze(1) + 1  # +1 because CTC blank=0; word_vocab convention: labels are 0-indexed here
    tgt_lens = torch.ones(len(batch), dtype=torch.long)
    return inputs, targets, lengths, tgt_lens

train_loader_word_ctc = DataLoader(train_dataset, batch_size=32, shuffle=True,
                                    collate_fn=word_level_collate, num_workers=2, pin_memory=True)
val_loader_word_ctc = DataLoader(val_dataset, batch_size=32, shuffle=False,
                                  collate_fn=word_level_collate, num_workers=2, pin_memory=True)

EPOCHS = 20
best_val_loss = float('inf')

for epoch in range(1, EPOCHS + 1):
    print(f"\n--- Epoch {epoch}/{EPOCHS} ---")
    model.train()
    train_loss, valid_batches = 0.0, 0

    for batch_inputs, batch_targets, in_lens, tgt_lens in tqdm(train_loader_word_ctc, desc="Training"):
        batch_inputs, batch_targets = batch_inputs.to(device), batch_targets.to(device)
        in_lens = in_lens.to(device)

        optimizer.zero_grad()
        log_probs, pooled_lens = model(batch_inputs, in_lens)
        # NOTE: pool(kernel=2) on a length-1 sequence -> pooled length 0. Word-level single-frame
        # samples can't go through the pooling designed for multi-frame sequences as-is; see the
        # note below the training loop for how to handle this.
        log_probs = log_probs.permute(1, 0, 2)  # (T, B, C) for CTCLoss
        loss = criterion(log_probs, batch_targets, pooled_lens.clamp(min=1), tgt_lens.to(device))

        if not torch.isnan(loss) and not torch.isinf(loss):
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=2.0)
            optimizer.step()
            train_loss += loss.item()
            valid_batches += 1

    avg_train_loss = train_loss / max(1, valid_batches)

    model.eval()
    val_loss, val_valid_batches = 0.0, 0
    with torch.no_grad():
        for batch_inputs, batch_targets, in_lens, tgt_lens in val_loader_word_ctc:
            batch_inputs, batch_targets = batch_inputs.to(device), batch_targets.to(device)
            in_lens = in_lens.to(device)
            log_probs, pooled_lens = model(batch_inputs, in_lens)
            log_probs = log_probs.permute(1, 0, 2)
            loss = criterion(log_probs, batch_targets, pooled_lens.clamp(min=1), tgt_lens.to(device))
            if not torch.isnan(loss) and not torch.isinf(loss):
                val_loss += loss.item()
                val_valid_batches += 1

    avg_val_loss = val_loss / max(1, val_valid_batches)
    scheduler.step(avg_val_loss)
    print(f"📉 Train Loss: {avg_train_loss:.4f} | 📈 Val Loss: {avg_val_loss:.4f}")

    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(model.state_dict(), "best_isl_conformer_word.pth")
        print("💾 New best model saved.")


### ⚠️ Important note on word-level + `ISL_Conformer`'s pooling
`ISL_Conformer` applies `MaxPool1d(kernel_size=2)` before the Conformer stack, halving sequence length. A single-frame word-level sample (`T=1`) pools down to `T=0`, which will break both the Conformer stack and CTC loss (zero-length target sequence). Two ways to actually make Stage 1 work with this architecture:

1. **Use short clips, not single images, for word-level too** — extract a few consecutive frames per word sample (e.g. 4–8 frames) instead of one static image, so `T≥2` survives pooling. This is usually the better choice for sign language anyway, since single frames lose all motion information that many signs depend on.
2. **Or repeat/pad the single frame** to a minimum length (e.g. tile it 4x) before feeding it to the model, purely so the pooling operation has something to work on — this is a hack that doesn't add real information, so option 1 is strongly preferred if the corpus's `Frames_Word_Level` folders contain multiple frames per word (check — if each word folder has multiple frame images, sample a short contiguous window instead of treating every single image as an independent sample).

## 13. TRAIN — Stage 2 (sentence-level, fine-tune from Stage 1 weights)

In [ ]:
if 'sentence_dataset' in locals() and len(sentence_dataset) > 0:
    # Fine-tune from Stage 1 weights: same ISL_Conformer, same input_dim=163, same vocab space.
    model_sentence = ISL_Conformer(input_dim=163, num_classes=num_classes_word).to(device)
    model_sentence.load_state_dict(torch.load("best_isl_conformer_word.pth", map_location=device))
    print("✅ Loaded Stage 1 (word-level) weights into Stage 2 model — schema-compatible by construction.")

    optimizer_s2 = optim.AdamW(model_sentence.parameters(), lr=1e-4, weight_decay=1e-4)  # lower LR for fine-tune
    scheduler_s2 = optim.lr_scheduler.ReduceLROnPlateau(optimizer_s2, mode='min', factor=0.5, patience=3)
    criterion_s2 = torch.nn.CTCLoss(blank=0, zero_infinity=True)

    EPOCHS_S2 = 15
    best_val_loss_s2 = float('inf')

    for epoch in range(1, EPOCHS_S2 + 1):
        print(f"\n--- [Sentence] Epoch {epoch}/{EPOCHS_S2} ---")
        model_sentence.train()
        train_loss, valid_batches = 0.0, 0

        for batch_inputs, batch_targets, in_lens, tgt_lens in tqdm(train_loader_sentence, desc="Training"):
            batch_inputs, batch_targets = batch_inputs.to(device), batch_targets.to(device)
            in_lens = in_lens.to(device)

            optimizer_s2.zero_grad()
            log_probs, pooled_lens = model_sentence(batch_inputs, in_lens)
            log_probs = log_probs.permute(1, 0, 2)
            loss = criterion_s2(log_probs, batch_targets, pooled_lens.clamp(min=1), tgt_lens.to(device))

            if not torch.isnan(loss) and not torch.isinf(loss):
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model_sentence.parameters(), max_norm=2.0)
                optimizer_s2.step()
                train_loss += loss.item()
                valid_batches += 1

        avg_train_loss = train_loss / max(1, valid_batches)

        model_sentence.eval()
        val_loss, val_valid_batches = 0.0, 0
        with torch.no_grad():
            for batch_inputs, batch_targets, in_lens, tgt_lens in val_loader_sentence:
                batch_inputs, batch_targets = batch_inputs.to(device), batch_targets.to(device)
                in_lens = in_lens.to(device)
                log_probs, pooled_lens = model_sentence(batch_inputs, in_lens)
                log_probs = log_probs.permute(1, 0, 2)
                loss = criterion_s2(log_probs, batch_targets, pooled_lens.clamp(min=1), tgt_lens.to(device))
                if not torch.isnan(loss) and not torch.isinf(loss):
                    val_loss += loss.item()
                    val_valid_batches += 1

        avg_val_loss = val_loss / max(1, val_valid_batches)
        scheduler_s2.step(avg_val_loss)
        print(f"📉 Train Loss: {avg_train_loss:.4f} | 📈 Val Loss: {avg_val_loss:.4f}")

        if avg_val_loss < best_val_loss_s2:
            best_val_loss_s2 = avg_val_loss
            torch.save(model_sentence.state_dict(), "best_isl_conformer_sentence.pth")
            print("💾 New best sentence-level model saved.")
else:
    print("⚠️ Sentence-level dataset is empty — fill in Section 6's annotation file and re-run "
          "Sections 6, 9, 11 before this cell.")


## 14. WER / sequence-level evaluation (NEW — did not exist before)
Word-level confusion-matrix eval only checked whether the model got the *first* word right. For sentence-level, we need actual sequence accuracy: Word Error Rate via edit distance.

In [ ]:
from itertools import groupby

def ctc_greedy_decode(log_probs_row):
    """log_probs_row: (T, C) log-probs for one sample. Returns list of predicted word ids (blank/dup collapsed)."""
    pred_ids = log_probs_row.argmax(dim=-1).cpu().numpy()
    return [k for k, _ in groupby(pred_ids) if k != 0]

def edit_distance(ref, hyp):
    """Standard Levenshtein distance between two token sequences."""
    n, m = len(ref), len(hyp)
    dp = [[0] * (m + 1) for _ in range(n + 1)]
    for i in range(n + 1): dp[i][0] = i
    for j in range(m + 1): dp[0][j] = j
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            cost = 0 if ref[i - 1] == hyp[j - 1] else 1
            dp[i][j] = min(dp[i - 1][j] + 1, dp[i][j - 1] + 1, dp[i - 1][j - 1] + cost)
    return dp[n][m]

def compute_wer(model, loader, device):
    model.eval()
    total_errors, total_ref_len = 0, 0
    examples = []
    with torch.no_grad():
        for batch_inputs, batch_targets, in_lens, tgt_lens in loader:
            batch_inputs = batch_inputs.to(device)
            in_lens = in_lens.to(device)
            log_probs, pooled_lens = model(batch_inputs, in_lens)

            for i in range(batch_inputs.size(0)):
                ref = batch_targets[i, :tgt_lens[i]].tolist()
                hyp = ctc_greedy_decode(log_probs[i])
                total_errors += edit_distance(ref, hyp)
                total_ref_len += len(ref)
                if len(examples) < 5:
                    examples.append((ref, hyp))

    wer = total_errors / max(1, total_ref_len)
    return wer, examples

if 'model_sentence' in locals() and 'val_loader_sentence' in locals():
    wer, examples = compute_wer(model_sentence, val_loader_sentence, device)
    print(f"📊 Sentence-level WER: {wer:.2%}")
    print("\nSample predictions (ref vs hyp, as word-ids):")
    for ref, hyp in examples:
        print(f"  ref={ref}  hyp={hyp}")
else:
    print("⚠️ Run Stage 2 training first.")


## 15. Save final model + vocab to Drive

In [ ]:
import os, json, shutil
from google.colab import drive

drive.mount('/content/drive')
drive_dir = "/content/drive/MyDrive/ISL_Model_Backup"
os.makedirs(drive_dir, exist_ok=True)

# Save whichever stage you've trained furthest (word-only or word+sentence fine-tuned)
final_model_path = "best_isl_conformer_sentence.pth" if os.path.exists("best_isl_conformer_sentence.pth") \
                    else "best_isl_conformer_word.pth"

if os.path.exists(final_model_path):
    shutil.copy(final_model_path, os.path.join(drive_dir, os.path.basename(final_model_path)))
    print(f"✅ Model backed up: {final_model_path}")
else:
    print("❌ No trained model checkpoint found yet.")

if 'class_to_idx' in locals():
    idx_to_word = {v: k for k, v in class_to_idx.items()}
    with open("vocab.json", "w") as f:
        json.dump(idx_to_word, f)
    shutil.copy("vocab.json", os.path.join(drive_dir, "vocab.json"))
    print("✅ Vocabulary backed up.")
else:
    print("❌ class_to_idx not found — run Section 9 first.")
